# Auto-inference demo

`run_auto_inference()` runs the full Joint SSMT pipeline from one call. Pass it an LFP array, a spike array, and a sampling rate. It prints a pre-flight summary, asks for one confirmation, and writes the inference results and the default plots to a single output directory.

The notebook covers both the single-trial case (`trial_structure=False`, default) and the trial-structured case (`trial_structure=True`).

Steps below:

1. load a dataset
2. run with defaults (single trial)
3. inspect the output directory
4. re-run with config overrides
5. trial-structured data


## 1. Load the demo dataset

A small dataset (180 s of LFP plus 2 spike units at 1 kHz) is bundled with the package and accessible through `load_demo_data()`. In a real workflow, replace this with your own LFP, spikes, and sampling rate.

In [1]:
import numpy as np
from joint_ssmt import load_demo_data

# load_demo_data() returns the bundled dataset shipped with the package, so this
# works whether you cloned the repo or just pip-installed.
data    = load_demo_data()
lfp     = data['lfp']
spikes  = data['spikes']
fs      = data['fs']

print(f"LFP:    shape {lfp.shape}, range [{lfp.min():.2f}, {lfp.max():.2f}]")
print(f"Spikes: shape {spikes.shape}, total spike count = {int(spikes.sum())}")
print(f"fs:     {fs:g} Hz, duration = {lfp.shape[0] / fs:.1f} s")


LFP:    shape (180000,), range [-11.35, 12.50]
Spikes: shape (2, 180000), total spike count = 16745
fs:     1000 Hz, duration = 180.0 s


/home/bowen27/miniconda3/envs/jax312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Run with defaults

`run_auto_inference()` prints the pre-flight summary and waits for Enter. To run end-to-end without the prompt (e.g. inside a notebook that you want to execute top-to-bottom), pass `interactive=False`.

In [2]:
from joint_ssmt import run_auto_inference

saved = run_auto_inference(
    lfp,
    spikes,
    fs,
    output_dir='./auto_results_default',
    interactive=False,   # set to True for the confirmation prompt
)


W0507 22:13:05.897145  809057 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0507 22:13:05.899449  802697 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


[EM-CT-JAX] iter    0  Q = -7.7448e+04
[EM-CT-JAX] Unfreezing lambda at iter 50
[EM-CT-JAX] iter  100  Q = 4.8292e+04
[EM-CT-JAX] iter  200  Q = 4.7890e+04
[EM-CT-JAX] iter  300  Q = 4.7679e+04
[EM-CT-JAX] iter  400  Q = 4.7572e+04
[EM-CT-JAX] iter  500  Q = 4.7523e+04
[EM-CT-JAX] iter  600  Q = 4.7497e+04
[EM-CT-JAX] iter  700  Q = 4.7481e+04
[EM-CT-JAX] iter  800  Q = 4.7469e+04
[EM-CT-JAX] iter  900  Q = 4.7463e+04
[EM-CT-JAX] iter 1000  Q = 4.7460e+04
[EM-CT-JAX] iter 1100  Q = 4.7458e+04
[EM-CT-JAX] iter 1200  Q = 4.7458e+04
[EM-CT-JAX] iter 1300  Q = 4.7458e+04
[EM-CT-JAX] iter 1400  Q = 4.7458e+04
[EM-CT-JAX] iter 1500  Q = 4.7458e+04
[EM-CT-JAX] iter 1600  Q = 4.7458e+04
[EM-CT-JAX] iter 1700  Q = 4.7458e+04
[EM-CT-JAX] iter 1800  Q = 4.7458e+04
[EM-CT-JAX] iter 1900  Q = 4.7458e+04
[EM-CT-JAX] iter 1999  Q = 4.7458e+04


Warmup (PG-Gibbs): 100%|██████████| 300/300 [00:33<00:00,  8.97it/s]


## 3. What got saved

The output directory contains:

- `coupling.npz`, `metadata.json`, optionally `spectral.npz`: the inference results.
- PNG/PDF files for the default summary plots.


In [3]:
from pathlib import Path
out = Path('./auto_results_default')
for f in sorted(out.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'{f.name:40s}  {size_kb:>8.1f} KB')


coupling.npz                                 540.9 KB
figures                                        0.0 KB
metadata.json                                  0.8 KB
spectral.npz                              1314713.8 KB


## 4. Override the defaults

Pass an `overrides` dict to change any setting. Top-level keys are `'spectral'`, `'inference'`, and `'output'`. Only the keys you specify are touched; everything else stays at the default.

Example: only analyze 1-30 Hz, use a longer multitaper window, and run more warmup iterations.

In [4]:
saved2 = run_auto_inference(
    lfp,
    spikes,
    fs,
    output_dir='./auto_results_lowfreq',
    overrides={
        'spectral':  {'freq_max': 30, 'window_sec': 4.0},
        'inference': {'warmup_iterations': 600},
    },
    interactive=False,
)


[EM-CT-JAX] iter    0  Q = -1.9332e+04
[EM-CT-JAX] Unfreezing lambda at iter 50
[EM-CT-JAX] iter  100  Q = 1.0382e+04
[EM-CT-JAX] iter  200  Q = 1.0553e+04
[EM-CT-JAX] iter  300  Q = 1.0546e+04
[EM-CT-JAX] iter  400  Q = 1.0564e+04
[EM-CT-JAX] iter  500  Q = 1.0571e+04
[EM-CT-JAX] iter  600  Q = 1.0573e+04
[EM-CT-JAX] iter  700  Q = 1.0573e+04
[EM-CT-JAX] iter  800  Q = 1.0573e+04
[EM-CT-JAX] iter  900  Q = 1.0573e+04
[EM-CT-JAX] Converged at iter 997  Q = 1.0573e+04


Warmup (PG-Gibbs): 100%|██████████| 600/600 [00:50<00:00, 11.90it/s]


## 5. Trial-structured data

For repeated-trial experiments, pass `trial_structure=True`. The wrapper expects `lfp` of shape `(R, T)` and `spikes` of shape `(R, S, T)` and dispatches to `run_inference_trials()` with the same configuration system.

To demonstrate without needing a separate trial-structured dataset, we split the demo recording into 6 contiguous segments of 30 seconds each and treat them as pseudo-trials. On real data you would just pass your trial-structured arrays directly.

In [5]:
R = 6
T_per_trial = lfp.shape[0] // R
T_use = R * T_per_trial   # in case duration is not exactly divisible

lfp_trials    = lfp[:T_use].reshape(R, T_per_trial)
spikes_trials = spikes[:, :T_use].reshape(spikes.shape[0], R, T_per_trial).transpose(1, 0, 2)

print(f'lfp_trials shape    = {lfp_trials.shape}    (R, T)')
print(f'spikes_trials shape = {spikes_trials.shape} (R, S, T)')

saved_trials = run_auto_inference(
    lfp_trials,
    spikes_trials,
    fs,
    output_dir='./auto_results_trials',
    trial_structure=True,
    interactive=False,
)


lfp_trials shape    = (6, 30000)    (R, T)
spikes_trials shape = (6, 2, 30000) (R, S, T)
[EM-CT-HIER-JAX] iter 0  Q=4.139615e+04
[EM-CT-HIER-JAX] iter 1000  Q=7.473484e+04


Warmup (trial PG-Gibbs): 100%|██████████| 300/300 [00:31<00:00,  9.50it/s]


[EM-CT-HIER-JAX] iter 0  Q=4.139615e+04


## 6. Interactive mode

Leave `interactive=True` (the default) for real use. The function prints the pre-flight summary, then waits for Enter to run or for the string `abort` to cancel. The point of the prompt is to catch obvious mistakes before a multi-minute MCMC run, so it is worth a glance: if the duration is off by an order of magnitude, the spike rates are zero, or the units count is unexpected, abort and check whether `lfp` and `spikes` have been transposed or truncated.

```python
run_auto_inference(lfp, spikes, fs)   # prompts once, then runs
```